# LeetCode #204 — Count Primes: sàng Eratosthenes

**Mức độ:** Medium · **Chủ đề:** Array, Math, Enumeration, Number Theory

Skeleton trên LeetCode:

```python
class Solution:
    def countPrimes(self, n: int) -> int:
```

Mục tiêu của notebook:

1. Kiểm tra một số nguyên tố bằng **chia thử tới `√x`** — và vì sao làm vậy cho từng số tới `5·10^6` là chậm.
2. **Sàng Eratosthenes**: thay vì hỏi "`x` có ước không", ta **gạch bỏ bội** của từng số nguyên tố.
3. Ba tối ưu: bắt đầu gạch từ `p²`, chỉ xét `p <= √n`, gạch bằng **gán lát cắt** (`bytearray`).
4. Mở rộng: sàng ước nguyên tố nhỏ nhất, sàng đoạn.

# 1. Bài toán là gì?

Cho số nguyên `n`, trả về số lượng **số nguyên tố nhỏ hơn hẳn** `n`.

(Số nguyên tố: số tự nhiên `> 1` chỉ chia hết cho 1 và chính nó.)

**Constraints (theo đề):**

- `0 <= n <= 5 * 10^6`

---

## Ví dụ

```text
n = 10  ->  4      (2, 3, 5, 7)
n = 0   ->  0
n = 1   ->  0
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

### Số nguyên tố và kiểm tra bằng thử chia

`p >= 2` là **số nguyên tố** nếu chỉ chia hết cho 1 và chính nó. Để kiểm tra `n`, chỉ cần thử các ước `d` với `d² <= n`: nếu `n = a · b` với `a <= b` thì `a <= √n`.

In [1]:
from math import isqrt, gcd

def is_prime_demo(n):
    if n < 2:
        return False
    for d in range(2, isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

print([x for x in range(30) if is_prime_demo(x)])
print("isqrt(10**12 + 1) =", isqrt(10**12 + 1), "(căn nguyên chính xác, không lỗi làm tròn số thực)")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
isqrt(10**12 + 1) = 1000000 (căn nguyên chính xác, không lỗi làm tròn số thực)


### Sàng Eratosthenes: mọi số nguyên tố `< n` trong O(n log log n)

Ý tưởng: đi từ 2 lên; gặp số `p` chưa bị gạch thì `p` nguyên tố, **gạch mọi bội** của nó. Bắt đầu gạch từ `p²` (các bội nhỏ hơn đã bị gạch bởi ước nhỏ hơn), và chỉ cần `p² < n`.

```text
2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
    x   x   x x  x     x     x  x  x     x       (gạch bội của 2 từ 4, của 3 từ 9)
```

In [2]:
def sieve_demo(n):
    if n < 3:
        return []
    is_p = bytearray([1]) * n
    is_p[0] = is_p[1] = 0
    for p in range(2, isqrt(n - 1) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(range(p * p, n, p)))
    return [i for i in range(n) if is_p[i]]

print(sieve_demo(30), len(sieve_demo(10**5)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] 9592


### Ước chung lớn nhất: thuật toán Euclid

`gcd(a, b) = gcd(b, a mod b)`, dừng khi `b = 0`. Số bước `O(log min(a, b))`. Python có sẵn `math.gcd`.

In [3]:
def gcd_demo(a, b):
    steps = []
    while b:
        steps.append((a, b))
        a, b = b, a % b
    return a, steps

g, st = gcd_demo(252, 105)
print("gcd(252, 105) =", g, "| các bước:", st, "| math.gcd:", gcd(252, 105))

gcd(252, 105) = 21 | các bước: [(252, 105), (105, 42), (42, 21)] | math.gcd: 21


Ứng dụng: các bước nhảy `+k (mod n)` xuất phát từ 0 đi qua đúng `n / gcd(n, k)` vị trí rồi quay lại 0 — chia các vị trí thành `gcd(n, k)` vòng.

In [4]:
n_demo, k_demo = 6, 4
start, cycle = 0, [0]
while (cycle[-1] + k_demo) % n_demo != start:
    cycle.append((cycle[-1] + k_demo) % n_demo)
print(f"n={n_demo}, k={k_demo}: vòng từ 0 = {cycle}, độ dài {len(cycle)} = n / gcd = {n_demo // gcd(n_demo, k_demo)}")

n=6, k=4: vòng từ 0 = [0, 4, 2], độ dài 3 = n / gcd = 3


### Lũy thừa của số nguyên tố p trong n! (công thức Legendre)

Số lần `p` xuất hiện trong phân tích `n!` là `⌊n/p⌋ + ⌊n/p²⌋ + ⌊n/p³⌋ + …`: `⌊n/p⌋` số chia hết cho `p` mỗi số góp 1, `⌊n/p²⌋` số chia hết cho `p²` góp thêm 1, …

In [5]:
from math import factorial

def legendre_demo(n, p):
    total, q = 0, p
    while q <= n:
        total += n // q
        q *= p
    return total

def count_p_brute(n, p):
    f, c = factorial(n), 0
    while f % p == 0:
        f //= p; c += 1
    return c

for n_, p_ in [(10, 2), (25, 5), (100, 3)]:
    print(f"lũy thừa của {p_} trong {n_}! = {legendre_demo(n_, p_)} (đếm trực tiếp: {count_p_brute(n_, p_)})")

lũy thừa của 2 trong 10! = 8 (đếm trực tiếp: 8)
lũy thừa của 5 trong 25! = 6 (đếm trực tiếp: 6)
lũy thừa của 3 trong 100! = 48 (đếm trực tiếp: 48)


### Lỗi hay gặp

- Kiểm tra nguyên tố tới `n` thay vì `√n` → chậm; dùng `int(n ** 0.5)` có thể lệch với số lớn → dùng `isqrt`.
- Sàng gạch từ `2p` thay vì `p²` (vẫn đúng nhưng chậm hơn); quên 0 và 1 không nguyên tố.
- Tính `n!` thật rồi đếm → số khổng lồ, rất chậm.

## 2.2 Áp dụng vào bài này

Đếm số nguyên tố `< n` với `n` tới `5·10^6`: thử chia từng số quá chậm, nên dùng **sàng Eratosthenes** — gạch bội của mỗi số nguyên tố bắt đầu từ `p²`, chỉ cần `p² < n`. Trong Python, gạch bằng **gán lát cắt** trên `bytearray` nhanh hơn nhiều so với vòng `for`.

# 3. Nền tảng

## Chia thử tới căn bậc hai

Nếu `x = a · b` với `a <= b` thì `a <= √x`. Vậy chỉ cần thử các ước `2..⌊√x⌋`; không có ước nào → nguyên tố.

In [6]:
import math, time, random

def is_prime(x: int) -> bool:
    if x < 2:
        return False
    i = 2
    while i * i <= x:
        if x % i == 0:
            return False
        i += 1
    return True

print([x for x in range(30) if is_prime(x)])
print("math.isqrt(99) =", math.isqrt(99), "(căn bậc hai làm tròn xuống, không qua số thực)")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
math.isqrt(99) = 9 (căn bậc hai làm tròn xuống, không qua số thực)


# 4. Baseline: kiểm tra từng số

In [7]:
def count_primes_trial(n: int) -> int:
    return sum(1 for x in range(2, n) if is_prime(x))

print(count_primes_trial(10), count_primes_trial(100))
t0 = time.time(); c = count_primes_trial(10**5); t1 = time.time()
print(f"n = 10^5: {c} số nguyên tố, mất {t1 - t0:.2f}s")

4

 25


n = 10^5: 9592 số nguyên tố, mất 0.57s


Tổng chi phí khoảng `O(n √n)`; với `n = 5·10^6` sẽ mất rất lâu. Ta cần cách làm "theo lô".

# 5. Ý tưởng chính: sàng Eratosthenes

Đảo câu hỏi: thay vì kiểm tra **từng** số có ước hay không, ta lấy **từng số nguyên tố** `p` và **gạch bỏ mọi bội** của nó (`2p, 3p, …`). Số nào **không bị gạch** là nguyên tố.

```text
2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19 20
p = 2: gạch 4, 6, 8, 10, 12, 14, 16, 18, 20
p = 3: gạch 9, 15 (6, 12, 18 đã gạch)
p = 4: đã bị gạch → không phải nguyên tố → bỏ qua
p = 5: 5·5 = 25 > 20 → dừng
còn lại: 2 3 5 7 11 13 17 19
```

**Ba tối ưu:**

1. **Bắt đầu từ `p²`**: các bội `k·p` với `k < p` đã bị gạch bởi một số nguyên tố nhỏ hơn (ước nhỏ nhất của `k`).
2. **Chỉ cần `p <= √n`**: hợp số `< n` luôn có ước nguyên tố `<= √n`.
3. **Gạch bằng lát cắt**: `is_p[p*p : n : p] = bytes(...)` — làm trong C, nhanh hơn vòng `for` Python nhiều lần.

In [8]:
def count_primes_sieve_loop(n: int) -> int:
    if n < 3:
        return 0
    is_p = [True] * n
    is_p[0] = is_p[1] = False
    p = 2
    while p * p < n:
        if is_p[p]:
            for m in range(p * p, n, p):      # gạch bội, bắt đầu từ p²
                is_p[m] = False
        p += 1
    return sum(is_p)

print(count_primes_sieve_loop(10), count_primes_sieve_loop(100))

4 25


In [9]:
def count_primes_sieve(n: int) -> int:
    if n < 3:
        return 0
    is_p = bytearray([1]) * n                 # 1 byte mỗi số
    is_p[0] = is_p[1] = 0
    for p in range(2, math.isqrt(n - 1) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(range(p * p, n, p)))    # gán cả lát cắt về 0
    return sum(is_p)

print(count_primes_sieve(10), count_primes_sieve(100))

4 25


# 6. Mô phỏng

In [10]:
def trace_sieve(n):
    is_p = [True] * n
    is_p[0] = is_p[1] = False
    p = 2
    while p * p < n:
        if is_p[p]:
            crossed = [m for m in range(p * p, n, p) if is_p[m]]
            for m in range(p * p, n, p):
                is_p[m] = False
            print(f"p = {p}: gạch mới {crossed}")
        else:
            print(f"p = {p}: đã bị gạch, bỏ qua")
        p += 1
    print(f"dừng vì {p}² = {p * p} >= {n}")
    print("nguyên tố <", n, ":", [i for i in range(n) if is_p[i]])

trace_sieve(30)

p = 2: gạch mới [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28]
p = 3: gạch mới [9, 15, 21, 27]
p = 4: đã bị gạch, bỏ qua
p = 5: gạch mới [25]
dừng vì 6² = 36 >= 30
nguyên tố < 30 : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


# 7. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def countPrimes(self, n: int) -> int:
        if n < 3:
            return 0
        is_p = bytearray([1]) * n
        is_p[0] = is_p[1] = 0
        for p in range(2, math.isqrt(n - 1) + 1):
            if is_p[p]:
                is_p[p * p::p] = bytes(len(range(p * p, n, p)))
        return sum(is_p)

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


sol = Solution()
check("n = 10", sol.countPrimes(10), 4)
check("n = 0", sol.countPrimes(0), 0)
check("n = 1", sol.countPrimes(1), 0)
check("n = 2", sol.countPrimes(2), 0)
check("n = 3", sol.countPrimes(3), 1)
check("n = 100", sol.countPrimes(100), 25)
check("n = 10^6", sol.countPrimes(10**6), 78498)

for n in range(0, 2000):
    assert sol.countPrimes(n) == count_primes_trial(n) == count_primes_sieve_loop(n), n
print("n = 0..1999: khớp đếm bằng chia thử")

for name, f in [("sàng + lát cắt", sol.countPrimes), ("sàng vòng for", count_primes_sieve_loop)]:
    t0 = time.time(); r = f(5 * 10**6); t1 = time.time()
    print(f"n = 5·10^6, {name:<14}: {r} số nguyên tố, {t1 - t0:.2f}s")

n = 10       PASS  4
n = 0        PASS  0
n = 1        PASS  0
n = 2        PASS  0
n = 3        PASS  1
n = 100      PASS  25


n = 10^6     PASS  78498


n = 0..1999: khớp đếm bằng chia thử
n = 5·10^6, sàng + lát cắt: 348513 số nguyên tố, 0.11s


n = 5·10^6, sàng vòng for : 348513 số nguyên tố, 1.06s


# 8. Vì sao đúng? Độ phức tạp?

- Một số `m` bị gạch ⟺ `m = k·p` với `p` nguyên tố và `k >= p` ⟹ `m` là hợp số.
- Ngược lại mọi hợp số `m < n` có ước nguyên tố nhỏ nhất `p` với `p² <= m < n` (vì `m = p · k`, `k >= p`) ⟹ `m` bị gạch khi xét `p`.
- Khi xét tới `p`, `is_p[p]` còn là 1 ⟺ `p` nguyên tố (mọi ước nguyên tố nhỏ hơn của nó đã gạch xong).

```text
Time:  O(n log log n)    (Σ n/p trên các số nguyên tố p ≤ √n)
Space: O(n) — 1 byte mỗi số với bytearray (5 MB cho n = 5·10^6)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Đếm cả `n` ("nhỏ hơn hẳn")

In [13]:
def wrong_inclusive(n):
    return sum(1 for x in range(2, n + 1) if is_prime(x))      # SAI: tính cả n

print("n = 7, sai:", wrong_inclusive(7), "| đúng:", sol.countPrimes(7))

n = 7, sai: 4 | đúng: 3


## Lỗi 2: Gạch bắt đầu từ `p` (gạch luôn chính số nguyên tố)

In [14]:
def wrong_start_at_p(n):
    if n < 3:
        return 0
    is_p = [True] * n
    is_p[0] = is_p[1] = False
    for p in range(2, n):
        if is_p[p]:
            for m in range(p, n, p):                # SAI: gạch cả p
                is_p[m] = False
    return sum(is_p)

print("n = 10, sai:", wrong_start_at_p(10), "| đúng:", sol.countPrimes(10))

n = 10, sai: 0 | đúng: 4


Bắt đầu từ `2p` thì đúng (chỉ chậm hơn); bắt đầu từ `p` thì gạch mất chính số nguyên tố → đếm được 0.

## Lỗi 3: Không xử lý `n < 3`

In [15]:
def wrong_small(n):
    is_p = [True] * n
    is_p[0] = is_p[1] = False                       # n = 0 hoặc 1 → IndexError
    return sum(is_p)

for n in [0, 1]:
    try:
        wrong_small(n)
    except IndexError as e:
        print(f"n = {n}: IndexError ({e})")

n = 0: IndexError (list assignment index out of range)
n = 1: IndexError (list assignment index out of range)


## Lỗi 4: Dùng `math.sqrt` và làm tròn sai

`int(math.sqrt(x))` qua số thực có thể lệch 1 ở số lớn (sai số dấu phẩy động). `math.isqrt` tính chính xác bằng số nguyên. Với bài này `n <= 5·10^6` thì `sqrt` vẫn đúng, nhưng tập thói quen dùng `isqrt`.

In [16]:
x = (10**8 + 7) ** 2 - 1
print("int(math.sqrt(x)) =", int(math.sqrt(x)), "| math.isqrt(x) =", math.isqrt(x), "| đúng là", 10**8 + 6)

int(math.sqrt(x)) = 100000007 | math.isqrt(x) = 100000006 | đúng là 100000006


# 10. Mẹo nhận diện pattern

- **Hỏi về số nguyên tố / ước số cho mọi số tới `n`** → sàng (gạch bội) thay vì kiểm tra từng số.
- **Cần phân tích thừa số nhiều số** → sàng **ước nguyên tố nhỏ nhất** (spf), mỗi lần phân tích `O(log x)`.
- **Số rất lớn nhưng đoạn ngắn** `[L, R]` → **sàng đoạn**: dùng các số nguyên tố `<= √R` gạch trong đoạn.

Bài liên quan: **#263 Ugly Number** (Easy), **#2523 Closest Prime Numbers in Range**, **#952 Largest Component Size by Common Factor** (Hard), **#1175 Prime Arrangements** (Easy), **#2601 Prime Subtraction Operation**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy sàng cho `n = 50`: những `p` nào được dùng để gạch? Liệt kê số nguyên tố.

## Bài 2 — Điền chỗ trống

In [17]:
def exercise_2_count_primes(n: int) -> int:
    if n < 3:
        return 0
    is_p = [True] * n
    is_p[0] = is_p[1] = False
    p = 2
    while p * p < n:
        # TODO: nếu p nguyên tố thì gạch các bội từ p*p
        p += 1
    return sum(is_p)


# assert exercise_2_count_primes(10) == 4
# assert exercise_2_count_primes(2) == 0
# assert exercise_2_count_primes(100) == 25

## Bài 3 — Biến thể: liệt kê các số nguyên tố `<= n`

In [18]:
def exercise_3_primes_upto(n: int) -> list:
    # TODO
    pass


# assert exercise_3_primes_upto(20) == [2, 3, 5, 7, 11, 13, 17, 19]
# assert exercise_3_primes_upto(1) == []
# assert exercise_3_primes_upto(2) == [2]

## Bài 4 — Biến thể: sàng ước nguyên tố nhỏ nhất (spf) và phân tích thừa số

Tạo `spf[x]` = ước nguyên tố nhỏ nhất của `x` (với `2 <= x <= n`). Sau đó phân tích `x` bằng cách lặp `x //= spf[x]`.

In [19]:
def exercise_4_factorize_all(n: int, queries: list) -> list:
    # TODO: trả về list các phân tích (list thừa số nguyên tố tăng dần, có lặp) cho từng số trong queries (2 <= q <= n)
    pass


# assert exercise_4_factorize_all(100, [12, 97, 64, 90]) == [[2, 2, 3], [97], [2, 2, 2, 2, 2, 2], [2, 3, 3, 5]]

## Bài 5 — Mini challenge: sàng đoạn — đếm số nguyên tố trong `[L, R]`

`1 <= L <= R <= 10^12`, `R − L <= 10^6`. Không thể sàng tới `10^12`; hãy sàng các số nguyên tố `<= √R` rồi dùng chúng gạch các bội trong đoạn `[L, R]`.

In [20]:
def exercise_5_count_primes_segment(L: int, R: int) -> int:
    # TODO
    pass


# assert exercise_5_count_primes_segment(1, 10) == 4
# assert exercise_5_count_primes_segment(10**12 - 100, 10**12) == 4
# assert exercise_5_count_primes_segment(2, 2) == 1

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_sieve(50)` sau khi tự làm.

**Bài 3**
- Mức 1: sàng tới `n` (mảng dài `n + 1`), trả các chỉ số còn đánh dấu.

**Bài 4**
- Mức 1: `spf = list(range(n + 1))`; với `p` từ 2 tới `√n`, nếu `spf[p] == p` thì với `m` từ `p*p` bước `p`: nếu `spf[m] == m` thì `spf[m] = p`.

**Bài 5**
- Mức 1: `base = primes_upto(isqrt(R))`; mảng `seg` dài `R − L + 1`.
- Mức 2: với mỗi `p`: bội đầu tiên `>= L` là `start = max(p * p, (L + p − 1) // p * p)`; gạch `seg[start − L :: p]`.
- Mức 3: nhớ `1` không phải nguyên tố (nếu `L == 1`).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
trace_sieve(50)

p = 2: gạch mới [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48]
p = 3: gạch mới [9, 15, 21, 27, 33, 39, 45]
p = 4: đã bị gạch, bỏ qua
p = 5: gạch mới [25, 35]
p = 6: đã bị gạch, bỏ qua
p = 7: gạch mới [49]
dừng vì 8² = 64 >= 50
nguyên tố < 50 : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


## Đáp án Bài 2

In [22]:
def answer_2_count_primes(n: int) -> int:
    return count_primes_sieve_loop(n)

assert answer_2_count_primes(10) == 4
assert answer_2_count_primes(2) == 0
assert answer_2_count_primes(100) == 25
print("Bài 2 PASS (giống count_primes_sieve_loop ở mục 5)")

Bài 2 PASS (giống count_primes_sieve_loop ở mục 5)


## Đáp án Bài 3

In [23]:
def answer_3_primes_upto(n: int) -> list:
    if n < 2:
        return []
    is_p = bytearray([1]) * (n + 1)
    is_p[0] = is_p[1] = 0
    for p in range(2, math.isqrt(n) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(range(p * p, n + 1, p)))
    return [i for i in range(n + 1) if is_p[i]]

assert answer_3_primes_upto(20) == [2, 3, 5, 7, 11, 13, 17, 19]
assert answer_3_primes_upto(1) == []
assert answer_3_primes_upto(2) == [2]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [24]:
def answer_4_factorize_all(n: int, queries: list) -> list:
    spf = list(range(n + 1))
    for p in range(2, math.isqrt(n) + 1):
        if spf[p] == p:
            for m in range(p * p, n + 1, p):
                if spf[m] == m:
                    spf[m] = p
    out = []
    for x in queries:
        f = []
        while x > 1:
            f.append(spf[x])
            x //= spf[x]
        out.append(f)
    return out

assert answer_4_factorize_all(100, [12, 97, 64, 90]) == [[2, 2, 3], [97], [2, 2, 2, 2, 2, 2], [2, 3, 3, 5]]
qs = list(range(2, 1001))
for x, f in zip(qs, answer_4_factorize_all(1000, qs)):
    assert math.prod(f) == x and all(is_prime(p) for p in f) and f == sorted(f)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
def answer_5_count_primes_segment(L: int, R: int) -> int:
    base = answer_3_primes_upto(math.isqrt(R))
    seg = bytearray([1]) * (R - L + 1)
    for p in base:
        start = max(p * p, (L + p - 1) // p * p)
        if start > R:
            continue
        seg[start - L::p] = bytes(len(range(start, R + 1, p)))
    if L == 1:
        seg[0] = 0
    return sum(seg)

assert answer_5_count_primes_segment(1, 10) == 4
assert answer_5_count_primes_segment(10**12 - 100, 10**12) == 4
assert answer_5_count_primes_segment(2, 2) == 1
for _ in range(500):
    L = random.randint(1, 3000); R = L + random.randint(0, 300)
    assert answer_5_count_primes_segment(L, R) == sum(is_prime(x) for x in range(L, R + 1))
t0 = time.time(); c5 = answer_5_count_primes_segment(10**12, 10**12 + 10**6); t1 = time.time()
print(f"[10^12, 10^12 + 10^6]: {c5} số nguyên tố, {t1 - t0:.2f}s")
print("Bài 5 PASS")

[10^12, 10^12 + 10^6]: 36249 số nguyên tố, 0.14s
Bài 5 PASS


# 14. Tổng kết

> Sàng Eratosthenes: đánh dấu mọi số là nguyên tố, với mỗi `p` từ 2 tới `√n` còn được đánh dấu thì gạch các bội từ `p²`. Đếm số còn lại. `O(n log log n)`; trong Python dùng `bytearray` + gán lát cắt cho nhanh. Nhớ "nhỏ hơn **hẳn** `n`" và `n < 3 → 0`.

---

## Checklist trước khi nộp

- [ ] `n < 3` → `0`?
- [ ] Không tính chính `n`?
- [ ] Gạch từ `p * p`, chỉ với `p <= √n`?
- [ ] Không gạch mất chính `p`?